In [ ]:
import json
import sqlite3

import pandas as pd
from openai import OpenAI

from paths import paths, prepare_preprocessing
from preprocessing.src.threading import parallelize

In [ ]:
prepare_preprocessing()

In [ ]:
client = OpenAI()

In [ ]:
CATEGORY_DEFINITIONS = """
    Categories for walkability-related amenities:
        'essentials': Everyday necessities – grocery, convenience, postbox, toilets, water, cash, pharmacies.
        'eating_out': Prepared food & drink – restaurants, cafés.
        'education': Learning venues – schools, universities, libraries, tutoring, childcare.
        'healthcare': Diagnosing, treating or preventing illness – clinics, dentists, hospitals, vets, pharmacies.
        'culture': Arts, heritage, religion, civic life – theatres, museums, places_of_worship, community_centres, memorials.
        'leisure': Relaxation & recreation – parks, ponds, lakes, benches, playgrounds, gyms, sports courts, cinemas, non-vice nightlife.
        'services': Direct customer-facing facilities offering tangible transactions or errands – hairdresser, repair shop, laundromat, bank ATM, post office. Excludes large agencies or regulatory bodies.
        'vice': Potentially harmful – fast-food, bar, pub, casino, betting, adult, vape, liquor stores.
    If a feature does not clearly belong to any of these categories (including, but not limited to, residential buildings, transport stops, boundaries, highways, and landuse), set 'ignore': true.
    Do not assign a category that is not in the list above.
"""

In [ ]:
# Annotate a single OSM key-value pair with a category and confidence score
def annotate_poi(key, value):
    prompt = f"""
    {CATEGORY_DEFINITIONS}

    Task:
        - Look at OpenStreetMap tag '{key} = {value}'.
        - Consider the description of the tag at wiki.openstreetmap.org/wiki/Map_features
        - Decide whether it should be ignored.
        - Otherwise, assign a primary category (and optionally one secondary) from the list above.
        - Give 0-100 integer confidence scores.
        - Reply STRICTLY with valid JSON conforming to this schema:
            {{
              'ignore': boolean,
              'primary': {{ 'category': str, 'score': int }},
              'secondary': {{ 'category': str, 'score': int }}?,
              'explanation': str  // ≤ 20 words
            }}
    """

    response = client.chat.completions.create(
        model = 'gpt-4o-2024-05-13',
        response_format = {'type': 'json_object'},
        messages = [
            {'role': 'system', 'content': 'You are an expert OpenStreetMap data annotator.'},
            {'role': 'user', 'content': prompt}
        ],
        temperature = 0.3,
    )

    content = response.choices[0].message.content

    try:
        result = json.loads(content)
    except json.JSONDecodeError as e:
        raise ValueError(f'Could not parse JSON: {e}\nRaw content was:\n{content}')

    return {'key': key, 'value': value, **result}

In [ ]:
# Read OSM tags from the retrieved SQLite database, filter them by key, and prepare for annotation
conn = sqlite3.connect(paths.raw.walkability.osm_tags)

valid_keys = [
    'aerialway',
    'aeroway',
    'amenity',
    'barrier',
    'boundary',
    'building',
    'craft',
    'emergency',
    'geological',
    'healthcare',
    'highway',
    'historic',
    'landuse',
    'leisure',
    'man made',
    'military',
    'natural',
    'office',
    'place',
    'power',
    'public transport',
    'railway',
    'route',
    'shop',
    'telecom',
    'tourism',
    'water',
    'waterway'
]

exclude_vals = {'yes', 'no', 'none', 'unknown', 'unclassified'}

sql = """
SELECT key, value, count_all
FROM project_unique_tags
WHERE count_all > 100
"""

osm_tags = (
    pd.read_sql_query(sql, conn)
    .query('key in @valid_keys')
    .loc[lambda df: ~df['value'].str.lower().isin(exclude_vals)]
    .sort_values('count_all', ascending = False)[['key', 'value']]
    .reset_index(drop = True)
)

In [ ]:
# Categorise all OSM tags
keys = osm_tags['key']
values = osm_tags['value']

results = parallelize(annotate_poi, keys, values)

In [ ]:
with open(paths.preprocessing.shared.osm_tags, 'w', encoding = 'utf-8') as f:
    json.dump(results, f, ensure_ascii = False, indent = 2)

In [ ]:
# Group results by primary category and save to JSON
grouped_json = (
    pd.DataFrame(results)
    .loc[lambda df: ~df['ignore']]
    .drop(columns = 'ignore')
    .assign(primary_category = lambda df: df['primary'].map(lambda d: d['category']))
    .groupby('primary_category')[['key', 'value']]
    .apply(lambda group: list(zip(group.key, group.value)))
    .to_dict()
)

output_path = paths.preprocessing.shared.grouped_tags

with open(output_path, 'w', encoding = 'utf-8') as f:
    json.dump(grouped_json, f, ensure_ascii = False, indent = 2)